# Delphi score bucketing

Compare a bucketed Delphi specification with the continuous-score model.

## Environment and imports

Run `uv sync --dev` from the project root before starting JupyterLab.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display
from sklearn.metrics import roc_auc_score

print(f"Notebook Python: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

## Find the data

In [2]:
DATA_PATH = Path(
    r"C:\Users\georg\OneDrive\Coding\Origination-Scorecard"
    r"\data\KGB Modelling\KGB in sample.csv"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

kgb = pd.read_csv(
    DATA_PATH,
    usecols=["account_id", "DelphiScore", "target_bad"],
    dtype={"account_id": "string"},
)

print(f"Loaded {len(kgb):,} accounts from:\n{DATA_PATH}")

Loaded 7,258 accounts from:
C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\KGB in sample.csv


## Validate columns

In [3]:
# Validate the existing in-sample dataset.
model_data = kgb[["DelphiScore", "target_bad"]].apply(pd.to_numeric, errors="coerce")

if not np.isfinite(model_data.to_numpy()).all():
    raise ValueError("DelphiScore or target_bad contains missing/invalid values.")

if not model_data["target_bad"].isin([0, 1]).all():
    raise ValueError("target_bad must contain only 0 and 1.")

if model_data["target_bad"].nunique() != 2:
    raise ValueError("Both good and bad accounts are required.")

if model_data["DelphiScore"].nunique() < 2:
    raise ValueError("DelphiScore must vary across accounts.")

y = model_data["target_bad"].astype(int)

X = sm.add_constant(
    model_data[["DelphiScore"]].astype(float),
    has_constant="add",
)

result = sm.Logit(y, X).fit(disp=False, maxiter=200)

if not result.mle_retvals["converged"]:
    raise RuntimeError("Logistic regression did not converge.")

predicted_pd = result.predict(X)
response_residuals = y - predicted_pd
ssr = float(np.square(response_residuals).sum())

In [5]:
bucket_data = model_data.copy()

# Equal-frequency buckets; fewer may be created if score ties prevent five.
bucket_data["Delphi bucket"] = pd.qcut(
    bucket_data["DelphiScore"],
    q=5,
    duplicates="drop",
)

bucket_summary = (
    bucket_data.groupby("Delphi bucket", observed=True)
    .agg(
        minimum_score=("DelphiScore", "min"),
        maximum_score=("DelphiScore", "max"),
        accounts=("target_bad", "size"),
        target_bads=("target_bad", "sum"),
        bad_rate=("target_bad", "mean"),
    )
    .reset_index()
)

bucket_summary["share_of_all_bads"] = (
    bucket_summary["target_bads"] / bucket_data["target_bad"].sum()
)

display(
    bucket_summary.style.format(
        {
            "minimum_score": "{:.0f}",
            "maximum_score": "{:.0f}",
            "accounts": "{:,.0f}",
            "target_bads": "{:,.0f}",
            "bad_rate": "{:.2%}",
            "share_of_all_bads": "{:.2%}",
        }
    )
)

,Delphi bucket,minimum_score,maximum_score,accounts,target_bads,bad_rate,share_of_all_bads
0,"(537.999, 770.0]",538,770,"1,474",73,4.95%,24.83%
1,"(770.0, 810.0]",771,810,"1,458",67,4.60%,22.79%
2,"(810.0, 842.0]",811,842,"1,448",53,3.66%,18.03%
3,"(842.0, 877.0]",843,877,"1,429",51,3.57%,17.35%
4,"(877.0, 999.0]",878,999,"1,449",50,3.45%,17.01%


This is a good smooth distribution in the buckets

---

In [6]:
# Fit a separate logistic regression using the existing Delphi buckets.
bucket_labels = bucket_data["Delphi bucket"].cat.categories

if len(bucket_labels) < 2:
    raise ValueError("At least two populated buckets are required.")

# Dummy variables: omit the lowest-score bucket as the reference.
X_bucket = pd.get_dummies(
    bucket_data["Delphi bucket"],
    drop_first=True,
    dtype=float,
)

X_bucket.columns = [
    f"Bucket {i}: {interval}" for i, interval in enumerate(bucket_labels[1:], start=2)
]

X_bucket = sm.add_constant(X_bucket, has_constant="add")
y_bucket = bucket_data["target_bad"].astype(int)

bucket_result = sm.Logit(y_bucket, X_bucket).fit(
    disp=False,
    maxiter=200,
)

if not bucket_result.mle_retvals["converged"]:
    raise RuntimeError("Bucket regression did not converge.")

bucket_predicted_pd = bucket_result.predict(X_bucket)
bucket_ssr = float(np.square(y_bucket - bucket_predicted_pd).sum())

# Coefficient results.
ci = bucket_result.conf_int(alpha=0.05)

bucket_coefficients = pd.DataFrame(
    {
        "Coefficient": bucket_result.params,
        "P-value": bucket_result.pvalues,
        "Z-statistic": bucket_result.tvalues,
        "95% CI lower": ci[0],
        "95% CI upper": ci[1],
        "Odds ratio": np.exp(bucket_result.params),
    }
).rename(index={"const": "Intercept (reference bucket)"})

# Model-level results.
bucket_statistics = pd.Series(
    {
        "McFadden pseudo-R²": bucket_result.prsquared,
        "Log-likelihood": bucket_result.llf,
        "Observed bad rate": y_bucket.mean(),
        "ROC AUC (in-sample)": roc_auc_score(y_bucket, bucket_predicted_pd),
        "AIC": bucket_result.aic,
        "Sum of squared response residuals": bucket_ssr,
    },
    name="Value",
)

print(f"Reference bucket: {bucket_labels[0]}")
display(bucket_coefficients)
display(bucket_statistics.to_frame())

# Export both tables into one CSV.
export_table = pd.concat(
    [
        bucket_coefficients.rename_axis("Term").reset_index().assign(Section="Coefficient"),
        bucket_statistics.rename_axis("Term").reset_index().assign(Section="Model statistic"),
    ],
    ignore_index=True,
)

output_path = DATA_PATH.parent / "Delphi bucket regression results.csv"
export_table.to_csv(output_path, index=False)

print(f"Results exported to:\n{output_path}")

Reference bucket: (537.999, 770.0]


,Coefficient,P-value,Z-statistic,95% CI lower,95% CI upper,Odds ratio
Intercept (reference bucket),-2.954482,9.851858e-134,-24.610085,-3.189779,-2.719185,0.052106
"Bucket 2: (770.0, 810.0]",-0.078603,6.502685e-01,-0.453389,-0.418400,0.261193,0.924406
"Bucket 3: (810.0, 842.0]",-0.315876,8.668541e-02,-1.713148,-0.677260,0.045509,0.729150
"Bucket 4: (842.0, 877.0]",-0.342081,6.648034e-02,-1.835171,-0.707423,0.023262,0.710291
"Bucket 5: (877.0, 999.0]",-0.377008,4.426847e-02,-2.011540,-0.744349,-0.009666,0.685911


,Value
McFadden pseudo-R²,0.002779
Log-likelihood,-1227.189697
Observed bad rate,0.040507
ROC AUC (in-sample),0.541643
AIC,2464.379393
Sum of squared response residuals,281.820394


Results exported to:
C:\Users\georg\OneDrive\Coding\Origination-Scorecard\data\KGB Modelling\Delphi bucket regression results.csv


**These buckets do not improve the model - keep continuous delphi**